In [77]:
# ============================================
# 02_merge_datasets.ipynb
# Merge season, weekly, and salary cap data
# into a unified dataset.
# ============================================

import pandas as pd
import os
os.chdir(r"C:\AI Projects\NFL Player Rating")
import pandas as pd

# Load season, weekly, and salary cap data
season_df = pd.read_csv("data/player_season/player_season_all.csv")
weekly_df = pd.read_csv("data/player_weekly/player_weekly_all.csv")
salary_df = pd.read_csv("data/salary_cap/salary_cap_all.csv")

print("Season:", season_df.shape)
print("Weekly:", weekly_df.shape)
print("Salary:", salary_df.shape)

Season: (3716, 58)
Weekly: (33287, 53)
Salary: (13260, 3)


In [78]:
# Inspect available columns
print("Weekly columns:", sorted(weekly_df.columns.tolist()))

# Build aggregation dict ONLY for columns that exist
agg_dict = {}

possible_stats = {
    "fantasy_points_ppr": "sum",
    "passing_yards": "sum",
    "rushing_yards": "sum",
    "receiving_yards": "sum",
    "passing_tds": "sum",
    "rushing_tds": "sum",
    "receiving_tds": "sum",
    "interceptions": "sum",
    "fumbles_lost": "sum",   # may or may not exist
    "fumbles": "sum",        # fallback
    "targets": "sum",
    "receptions": "sum",
    "attempts": "sum"
}

# Only include columns that actually exist in weekly_df
for col, func in possible_stats.items():
    if col in weekly_df.columns:
        agg_dict[col] = func

print("Using these weekly stats:", agg_dict)

# Aggregate weekly stats
weekly_agg = (
    weekly_df
    .groupby(["player_id", "season"])
    .agg(agg_dict)
    .reset_index()
)

weekly_agg.head()

Weekly columns: ['air_yards_share', 'attempts', 'carries', 'completions', 'dakota', 'fantasy_points', 'fantasy_points_ppr', 'headshot_url', 'interceptions', 'opponent_team', 'pacr', 'passing_2pt_conversions', 'passing_air_yards', 'passing_epa', 'passing_first_downs', 'passing_tds', 'passing_yards', 'passing_yards_after_catch', 'player_display_name', 'player_id', 'player_name', 'position', 'position_group', 'racr', 'receiving_2pt_conversions', 'receiving_air_yards', 'receiving_epa', 'receiving_first_downs', 'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_tds', 'receiving_yards', 'receiving_yards_after_catch', 'recent_team', 'receptions', 'rushing_2pt_conversions', 'rushing_epa', 'rushing_first_downs', 'rushing_fumbles', 'rushing_fumbles_lost', 'rushing_tds', 'rushing_yards', 'sack_fumbles', 'sack_fumbles_lost', 'sack_yards', 'sacks', 'season', 'season_type', 'special_teams_tds', 'target_share', 'targets', 'week', 'wopr']
Using these weekly stats: {'fantasy_points_ppr': 'sum', 

,player_id,season,fantasy_points_ppr,passing_yards,rushing_yards,receiving_yards,passing_tds,rushing_tds,receiving_tds,interceptions,targets,receptions,attempts
0,00-0019596,2019,270.04,4266.0,34.0,0.0,24,3,0,9.0,0,0,650
1,00-0019596,2020,420.06,5694.0,3.0,0.0,50,4,0,15.0,0,0,748
2,00-0019596,2021,406.74,5916.0,81.0,0.0,46,2,0,13.0,0,0,810
3,00-0019596,2022,293.70,5045.0,-1.0,0.0,27,1,0,10.0,1,0,799
4,00-0020531,2019,233.58,3187.0,1.0,0.0,28,1,0,5.0,0,0,411


In [79]:
# -----------------------------
# Fix salary cap player name column BEFORE merging
# -----------------------------

# The first column is always the player name, even if labeled "player (120)" etc.
first_col = salary_df.columns[0]
salary_df = salary_df.rename(columns={first_col: "player_name"})

# Ensure season is int
if "season" in salary_df.columns:
    salary_df["season"] = salary_df["season"].astype(int)

print("Salary DF columns after rename:", salary_df.columns.tolist())
salary_df.head()

Salary DF columns after rename: ['player_name', 'team', 'season']


,player_name,team,season
0,Jones Chandler Jones,arizona-cardinals,2019
1,Fitzgerald Larry Fitzgerald,arizona-cardinals,2019
2,Johnson David Johnson,arizona-cardinals,2019
3,Humphries D.J. Humphries,arizona-cardinals,2019
4,Peterson Patrick Peterson,arizona-cardinals,2019


In [80]:
# -----------------------------
# Merge season + weekly data
# -----------------------------
merged = pd.merge(
    season_df,
    weekly_agg,
    on=["player_id", "season"],
    how="inner"
)

# -----------------------------
# Add player_name, position, team from weekly_df
# -----------------------------
player_info = weekly_df[[
    "player_id",
    "player_name",
    "position",
    "recent_team"
]].drop_duplicates()

merged = pd.merge(
    merged,
    player_info,
    on="player_id",
    how="left"
)

# -----------------------------
# Merge salary cap data
# -----------------------------
merged = pd.merge(
    merged,
    salary_df,
    on=["player_name", "season"],
    how="left"
)

print("Merged with salary:", merged.shape)
merged.head()

Merged with salary: (7144, 73)


,player_id,season,season_type,completions,attempts_x,passing_yards_x,passing_tds_x,interceptions_x,sacks,sack_yards,...,rushing_tds_y,receiving_tds_y,interceptions_y,targets_y,receptions_y,attempts_y,player_name,position,recent_team,team
0,00-0019596,2019,REG,373,613,4057.0,24,8.0,27.0,185.0,...,3,0,9.0,0,0,650,T.Brady,QB,NE,NaN
1,00-0019596,2019,REG,373,613,4057.0,24,8.0,27.0,185.0,...,3,0,9.0,0,0,650,T.Brady,QB,TB,NaN
2,00-0020531,2019,REG,281,378,2979.0,27,4.0,12.0,89.0,...,1,0,5.0,0,0,411,D.Brees,QB,NO,NaN
3,00-0021206,2019,REG,3,5,24.0,0,0.0,0.0,0.0,...,0,0,0.0,0,0,29,J.McCown,QB,PHI,NaN
4,00-0022127,2019,REG,0,0,0.0,0,0.0,0.0,0.0,...,0,4,0.0,83,63,0,J.Witten,TE,DAL,NaN


In [81]:
# -----------------------------
# Light Cleaning for merged dataset
# -----------------------------

# 1. Standardize player_name formatting
merged["player_name"] = (
    merged["player_name"]
    .astype(str)
    .str.strip()
)

# 2. Ensure season is integer
merged["season"] = merged["season"].astype(int)

# 3. Drop duplicate rows
merged = merged.drop_duplicates()

# 4. Remove rows with missing player_name (Spotrac sometimes includes blank rows)
merged = merged[merged["player_name"].notna() & (merged["player_name"] != "")]

# 5. Clean salary columns if they exist
if "cap hit" in merged.columns:
    merged["cap hit"] = (
        merged["cap hit"]
        .astype(str)
        .str.replace(r"[\$,]", "", regex=True)
        .replace("", "0")
        .astype(float)
    )

if "cap hit pct  league cap" in merged.columns:
    merged["cap hit pct  league cap"] = (
        merged["cap hit pct  league cap"]
        .astype(str)
        .str.replace("%", "", regex=False)
        .replace("", "0")
        .astype(float)
    )

print("After cleaning:", merged.shape)
merged.head()

After cleaning: (7144, 73)


,player_id,season,season_type,completions,attempts_x,passing_yards_x,passing_tds_x,interceptions_x,sacks,sack_yards,...,rushing_tds_y,receiving_tds_y,interceptions_y,targets_y,receptions_y,attempts_y,player_name,position,recent_team,team
0,00-0019596,2019,REG,373,613,4057.0,24,8.0,27.0,185.0,...,3,0,9.0,0,0,650,T.Brady,QB,NE,NaN
1,00-0019596,2019,REG,373,613,4057.0,24,8.0,27.0,185.0,...,3,0,9.0,0,0,650,T.Brady,QB,TB,NaN
2,00-0020531,2019,REG,281,378,2979.0,27,4.0,12.0,89.0,...,1,0,5.0,0,0,411,D.Brees,QB,NO,NaN
3,00-0021206,2019,REG,3,5,24.0,0,0.0,0.0,0.0,...,0,0,0.0,0,0,29,J.McCown,QB,PHI,NaN
4,00-0022127,2019,REG,0,0,0.0,0,0.0,0.0,0.0,...,0,4,0.0,83,63,0,J.Witten,TE,DAL,NaN


In [82]:
merged.to_csv("data/merged_dataset.csv", index=False)
print("Saved merged_dataset.csv")

Saved merged_dataset.csv
